# unify_pums.ipynb

This notebook reads in the household and person-level PUMS for a given year, merges them, cleans up the ORIGIN/CHOSEN fields, and injects fields that will be used later on during modeling.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("../.."))


In [ ]:
year = 2018
path = "us/pums_2018_raw.csv"

In [ ]:
# Read only the columns this notebook actually uses -- 33 of the 300 in the IPUMS
# extract.
#
# This list is exhaustive for the notebook as written. Adding a variable downstream
# means adding it here too, otherwise it will KeyError rather than silently go missing.
USECOLS = [
    # identity / weight
    "CBSERIAL",
    "PERNUM",
    "PERWT",
    # geography and the origin/destination outcome
    "STATEFIP",
    "PUMA",
    "MIGPLAC1",
    "MIGPUMA1",
    "MIGPUMANOW",
    # decision-unit construction
    "SUBFAM",
    "SFRELATE",
    "RELATE",
    "RELATED",
    "AGE",
    # per-member covariates (SHARED_COLS)
    "GRADEATT",
    "RACE",
    "HISPAN",
    "BPL",
    "CITIZEN",
    "INDNAICS",
    "CLASSWKR",
    "MARST",
    "DIVINYR",
    "WIDINYR",
    "MARRINYR",
    "VETSTATD",
    "EMPSTAT",
    "EDUC",
    "EDUCD",
    "RACAMIND",
    "RACASIAN",
    "RACBLK",
    "RACPACIS",
    "RACWHT",
    "RACOTHER",
    "RENTGRS",
    "HHINCOME",
]


df = pd.read_csv(path, usecols=USECOLS)
df

ValueError: Usecols do not match columns, columns expected but not found: ['FWAGE1']

In [ ]:
df["PID"] = df["CBSERIAL"] * 1_000 + df["PERNUM"]
assert df["PID"].is_unique
df = df.set_index("PID")

In [ ]:
df["MIGPLAC1"].value_counts()

MIGPLAC1
0      2788765
6        46187
48       36447
12       27232
36       20896
        ...   
330         75
350         73
520         71
623         66
622         61
Name: count, Length: 108, dtype: int64

In [ ]:
df["PUMA"]

PID
2018010000049001    1600
2018010000058001    1900
2018010000219001    2000
2018010000246001    2400
2018010000251001    2701
                    ... 
2018001400326004     400
2018001400326005     400
2018001400502001     100
2018001400502002     100
2018001400515001     500
Name: PUMA, Length: 3214539, dtype: int64

In [ ]:
# origin is the MIGSP + MIGPUMA
# need ints since it is treated as a float by default
df["ORIGIN"] = df["MIGPLAC1"].astype(int).astype(str).str.zfill(2) + df[
    "MIGPUMA1"
].astype(int).astype(str).str.zfill(5)  # migpuma geography

# chosen is the current location, ST + PUMA
df["CHOSEN"] = df["STATEFIP"].astype(int).astype(str).str.zfill(2) + df["PUMA"].astype(
    int
).astype(str).str.zfill(5)  # puma geography

df["CHOSEN_MIGPUMA"] = df["STATEFIP"].astype(int).astype(str).str.zfill(2) + df[
    "MIGPUMANOW"
].astype(int).astype(str).str.zfill(5)

In [ ]:
df["ORIGIN"].value_counts()

ORIGIN
0000000     2788765
0603700       10810
0400100        5848
1703400        5806
2500390        5542
             ...   
33000001         75
35000001         73
52000001         71
62300001         66
62200001         61
Name: count, Length: 1039, dtype: int64

In [ ]:
# backfill the stay origins to the MIGPUMA where they are currently (chosen == origin)
df["ORIGIN"] = np.where(
    df["ORIGIN"] == "0000000",
    df["CHOSEN_MIGPUMA"],
    df["ORIGIN"],
)
# fill in the origin state with this backfill in place
df["ORIGIN_STATE"] = np.where(
    df["ORIGIN"].str.len() == 7, df["ORIGIN"].str[:2].astype(int), 999
)

# define STAY as moving outside the MIGPUMA
df["STAY"] = np.where(df["CHOSEN_MIGPUMA"] == df["ORIGIN"], 1, 0)

In [ ]:
df["ORIGIN"].value_counts()

ORIGIN
0603700     102203
2500390      49638
1703400      41902
0400100      41174
4804600      36741
             ...  
33000001        75
35000001        73
52000001        71
62300001        66
62200001        61
Name: count, Length: 1038, dtype: int64

In [ ]:
df["ORIGIN_STATE"].value_counts()

ORIGIN_STATE
6      377274
48     265623
12     199113
36     197113
42     128757
17     126968
39     118907
37     101294
13      99777
26      99181
34      88714
51      84057
53      75492
25      69599
4       68702
18      67427
47      67323
29      62272
24      59648
55      59634
27      55797
8       55563
45      49125
1       47512
21      45229
22      43570
41      41714
40      37582
9       36349
19      32195
49      31177
5       30415
20      29519
28      29075
32      28499
31      19450
35      19155
54      18106
999     17841
16      16517
15      14390
33      13648
23      13117
44      10325
30      10287
46       9075
10       9056
38       7882
2        6858
11       6510
50       6388
56       5738
Name: count, dtype: int64

In [ ]:
df["CHOSEN"].value_counts()

CHOSEN
0102500    4550
5310200    4083
5500700    4082
5500100    3958
1200500    3421
           ... 
2701503     580
5541001     575
2701403     566
2701402     523
4203207     509
Name: count, Length: 2351, dtype: int64

In [ ]:
df["STAY"].value_counts()

STAY
1    3025964
0     188575
Name: count, dtype: int64

In [ ]:
for c in df.columns:
    print(c)

CBSERIAL
STATEFIP
PUMA
RENTGRS
HHINCOME
PERNUM
PERWT
SUBFAM
SFRELATE
RELATE
RELATED
AGE
MARST
MARRINYR
DIVINYR
WIDINYR
RACE
HISPAN
BPL
CITIZEN
RACAMIND
RACASIAN
RACBLK
RACPACIS
RACWHT
RACOTHER
EDUC
EDUCD
GRADEATT
EMPSTAT
CLASSWKR
INDNAICS
MIGPLAC1
MIGPUMA1
MIGPUMANOW
VETSTATD
ORIGIN
CHOSEN
CHOSEN_MIGPUMA
ORIGIN_STATE
STAY


In [ ]:
df["IS_CHILD_UNDER_6"] = np.where(df["AGE"] < 6, 1, 0)
df["IS_CHILD_6_TO_17"] = np.where((df["AGE"] >= 6) & (df["AGE"] <= 17), 1, 0)
df["IS_65_OR_OLDER"] = np.where(df["AGE"] >= 65, 1, 0)
df["IN_LF"] = np.where(df["EMPSTAT"].isin([1, 2]), 1, 0)
df["WORKING"] = np.where(df["EMPSTAT"] == 1, 1, 0)

In [ ]:
PRIMARY = {1, 2}
df["UNIT"] = np.where(
    # sometimes subfam != 0 while sfrelate == 0
    (df.SUBFAM != 0) & (df.SFRELATE != 0),
    # subfamily → its own unit
    df.CBSERIAL.astype(str) + "_SF" + df.SUBFAM.astype(str),
    np.where(
        # count primary
        df.RELATE.isin(PRIMARY)
        # count related children don't count unrelated children, count ofster children
        | ((df.AGE < 18) & ((df.RELATE < 11) | (df.RELATED.isin([1242]))))
        # unmarried partner
        | df.RELATED.isin([1114]),
        # primary family, non-subfamily or child with age < 18
        df.CBSERIAL.astype(str) + "_P",
        # treat everyone else as singletons, individual decision units
        df.CBSERIAL.astype(str) + "_I" + df.PERNUM.astype(int).astype(str),
    ),
)

In [ ]:
df["_REF_PRIORITY"] = np.where((df["SFRELATE"] == 1) | (df["RELATE"] == 1), 0, 1)
df["_SEC_PRIORITY"] = np.where(
    (df["SFRELATE"] == 2) | (df["RELATE"] == 2) | (df["RELATED"] == 1114), 0, 1
)

units = df.groupby("UNIT").agg(
    REF_INDEX=("_REF_PRIORITY", "idxmin"),
    SEC_INDEX=("_SEC_PRIORITY", "idxmin"),
    NUM_CHILDREN_UNDER_6=("IS_CHILD_UNDER_6", "sum"),
    NUM_CHILDREN_6_TO_17=("IS_CHILD_6_TO_17", "sum"),
    SIZE=("_REF_PRIORITY", "size"),
    NUM_IN_IF=("IN_LF", "sum"),
    NUM_WORKING=("WORKING", "sum"),
    RENTGRS=("RENTGRS", "first"),
    HHINCOME=("HHINCOME", "first"),
)

has_sec = df.groupby("UNIT")["_SEC_PRIORITY"].min().eq(0)
units["SEC_INDEX"] = units["SEC_INDEX"].where(has_sec, units["REF_INDEX"])

In [ ]:
CATEGORIES = ["REF", "SEC"]

In [ ]:
SHARED_COLS = [
    "GRADEATT",
    "RACE",
    "HISPAN",
    "BPL",
    "CITIZEN",
    "WORKING",
    "INDNAICS",
    "CLASSWKR",
    "AGE",
    "MARST",
    "DIVINYR",
    "WIDINYR",
    "MARRINYR",
    "VETSTATD",
    "EMPSTAT",
    "RACAMIND",
    "RACASIAN",
    "RACBLK",
    "RACPACIS",
    "RACWHT",
    "RACOTHER",
    "EDUC",
    "EDUCD",
    "RELATE",
]
BASE_COLS = ["PERWT", "CHOSEN", "ORIGIN", "STAY", "STATEFIP", "ORIGIN_STATE"]

for c in BASE_COLS:
    units[c] = df.loc[units["REF_INDEX"].values, c].values

for c in SHARED_COLS:
    for category in CATEGORIES:
        units[f"{c}_{category}"] = df.loc[units[f"{category}_INDEX"].values, c].values

In [ ]:
mask = (
    (units["AGE_REF"] >= 18)
    & (units["AGE_SEC"] >= 18)
    # do not consider institutional inmates
    & (units["RELATE_REF"] != 13)
    & (units["ORIGIN_STATE"] <= 56)
    & (~units["ORIGIN_STATE"].isin([2, 15]))
    & (units["STATEFIP"] <= 56)
    & (~units["STATEFIP"].isin([2, 15]))
)

print(units.shape)
units_subset = units.loc[mask].copy()
print(units_subset.shape)

(1845792, 63)
(1739030, 63)


In [ ]:
units_subset["AGE_REF"].value_counts()

AGE_REF
18    44051
19    42003
20    37117
21    34246
60    31266
      ...  
95     3540
93     2540
92     2501
91     1525
96      172
Name: count, Length: 79, dtype: int64

In [ ]:
units_subset["PAIRED_UNIT"] = np.where(
    units_subset["REF_INDEX"] != units_subset["SEC_INDEX"], 1, 0
)

In [ ]:
units_subset["CHILD_UNDER_6"] = np.where(units_subset["NUM_CHILDREN_UNDER_6"] > 0, 1, 0)
units_subset["CHILD_6_TO_17"] = np.where(units_subset["NUM_CHILDREN_6_TO_17"] > 0, 1, 0)
units_subset["CHILD"] = np.where(
    (units_subset["CHILD_UNDER_6"] == 1) | (units_subset["CHILD_6_TO_17"] == 1), 1, 0
)
units_subset["NUM_CHILDREN"] = (
    units_subset["NUM_CHILDREN_UNDER_6"] + units_subset["NUM_CHILDREN_6_TO_17"]
)

In [ ]:
# Earner counts. WORK1 means "exactly one earner in the unit" -- for a couple that is
# XOR, for a single-person unit it is simply whether that person works. Previously
# WORK1 was 1 for EVERY unpaired unit regardless of employment (only 54% of which
# actually had a working reference person), and PAIR_WORK1 was identically 1 for all
# rows, so it carried no information at all.
_w_ref = units_subset["WORKING_REF"].astype(bool)
_w_sec = units_subset["WORKING_SEC"].astype(bool)
_paired = units_subset["PAIRED_UNIT"] == 1

units_subset["WORK2"] = np.where(_paired & _w_ref & _w_sec, 1, 0)
units_subset["WORK1"] = np.where(np.where(_paired, _w_ref ^ _w_sec, _w_ref), 1, 0)
# couples with exactly one earner -- the classic tied-mover case
units_subset["PAIR_WORK1"] = np.where(_paired & (units_subset["WORK1"] == 1), 1, 0)

units_subset["SINGLE_UNIT_WITH_CHILD"] = np.where(
    (units_subset["PAIRED_UNIT"] == 0) & (units_subset["CHILD"] == 1), 1, 0
)

In [ ]:
# Education brackets. IPUMS EDUC is a coarse 0-11 scale and merges "12th grade, no
# diploma" with high-school graduates, so the brackets are built from EDUCD (detailed,
# 0-116), which maps 1:1 onto the ACS SCHL categories this pipeline used before:
#   <=61  less than a HS diploma (incl. 61 = 12th grade, no diploma)
#   62-64 HS diploma or GED
#   65-90 some college / associate's
#   101   bachelor's
#   >=110 graduate or professional
units_subset["MAX_EDUC"] = units_subset[["EDUC_REF", "EDUC_SEC"]].max(axis=1)
units_subset["MAX_EDUCD"] = units_subset[["EDUCD_REF", "EDUCD_SEC"]].max(axis=1)

_e = units_subset["MAX_EDUCD"]
units_subset["EDU_NOHIGH"] = np.where(_e <= 61, 1, 0)
units_subset["EDU_ONLY_HIGH"] = np.where(_e.isin([62, 63, 64]), 1, 0)
units_subset["EDU_SOME_COLLEGE"] = np.where(_e.isin([65, 70, 71, 80, 81, 90]), 1, 0)
units_subset["EDU_ONLY_BACHELORS"] = np.where(_e == 101, 1, 0)
units_subset["EDU_GRADUATE_DEG"] = np.where(_e >= 110, 1, 0)
units_subset["EDU_BACHELORS_OR_HIGHER"] = np.where(_e >= 101, 1, 0)
units_subset["EDU_HAS_DEGREE"] = np.where(_e >= 101, 1, 0)
units_subset["EDU_NO_DEGREE"] = np.where(_e < 101, 1, 0)
units_subset["EDU_HIGH_BUT_NOT_BACHELORS"] = np.where((_e >= 62) & (_e < 101), 1, 0)

# the five mutually exclusive brackets must partition every unit -- create_estdata
# gathers OWN_EARNINGS_10K_BY_EDU off exactly one of them
_excl = [
    "EDU_NOHIGH",
    "EDU_ONLY_HIGH",
    "EDU_SOME_COLLEGE",
    "EDU_ONLY_BACHELORS",
    "EDU_GRADUATE_DEG",
]
assert (units_subset[_excl].sum(axis=1) == 1).all(), (
    "EDU_* brackets do not partition the units"
)

In [ ]:
# MEAN_AGE is the mean of two integers, so it can land on x.5. Upper edges are
# exclusive (< 35, < 65) rather than <= 34 / <= 64, otherwise units at 34.5 and 64.5
# fall into no bracket at all -- 13,919 of them before this fix.
units_subset["MEAN_AGE"] = units_subset[["AGE_REF", "AGE_SEC"]].mean(axis=1)
units_subset["AGE_UNDER_18"] = np.where(units_subset["MEAN_AGE"] < 18, 1, 0)
units_subset["AGE_18_34"] = np.where(
    (units_subset["MEAN_AGE"] >= 18) & (units_subset["MEAN_AGE"] < 35), 1, 0
)
units_subset["AGE_35_64"] = np.where(
    (units_subset["MEAN_AGE"] >= 35) & (units_subset["MEAN_AGE"] < 65), 1, 0
)
units_subset["AGE_OVER_65"] = np.where(units_subset["MEAN_AGE"] >= 65, 1, 0)

units_subset["AGE_18_22"] = np.where(
    (units_subset["MEAN_AGE"] >= 18) & (units_subset["MEAN_AGE"] < 23), 1, 0
)
units_subset["AGE_23_29"] = np.where(
    (units_subset["MEAN_AGE"] >= 23) & (units_subset["MEAN_AGE"] < 30), 1, 0
)
units_subset["AGE_30_39"] = np.where(
    (units_subset["MEAN_AGE"] >= 30) & (units_subset["MEAN_AGE"] < 40), 1, 0
)
units_subset["AGE_40_49"] = np.where(
    (units_subset["MEAN_AGE"] >= 40) & (units_subset["MEAN_AGE"] < 50), 1, 0
)
units_subset["AGE_50_64"] = np.where(
    (units_subset["MEAN_AGE"] >= 50) & (units_subset["MEAN_AGE"] < 65), 1, 0
)

# create_estdata asserts this partition holds
assert (
    units_subset[["AGE_UNDER_18", "AGE_18_34", "AGE_35_64", "AGE_OVER_65"]].sum(axis=1)
    == 1
).all(), "AGE_* brackets do not partition the units"

In [ ]:
# foreign born = not a US citizen at birth. IPUMS CITIZEN: 0 born in the US,
# 1 born abroad of American parents (still native), 2 naturalized, 3+ not a citizen.
# NB: this used BPL_SEC, where BPL is birthplace and 1 = Alabama -- so "BPL >= 2" was
# true for 98.5% of units and FOREIGN_BORN was almost always 1.
units_subset["FOREIGN_BORN"] = np.where(
    (units_subset["CITIZEN_REF"] >= 2) | (units_subset["CITIZEN_SEC"] >= 2), 1, 0
)

In [ ]:
units_subset["IN_COLLEGE"] = np.where(
    (units_subset["GRADEATT_REF"] >= 6) | (units_subset["GRADEATT_SEC"] >= 6), 1, 0
)

In [ ]:
# this doesn't match the true MARST in df because some of the MARST people do not represent a decision unit
# this is true for most of these
units_subset["MARRIED"] = np.where(
    units_subset["MARST_REF"].isin([1, 2]) & units_subset["MARST_SEC"].isin([1, 2]),
    1,
    0,
)
# married unit and there are actually 2 people
units_subset["MARRIED_AND_TOGETHER"] = np.where(
    units_subset["MARRIED"] & (units_subset["PAIRED_UNIT"] == 1), 1, 0
)
# defined as either the reference or secondary person being divorced/widowed
units_subset["RECENTLY_WIDOWED_OR_DIVORCED"] = np.where(
    (units_subset["DIVINYR_REF"] == 2)
    | (units_subset["WIDINYR_REF"] == 2)
    | (units_subset["DIVINYR_SEC"] == 2)
    | (units_subset["WIDINYR_SEC"] == 2),
    1,
    0,
)
# referring to a recently married couple (or one of them, if they are living alone)
units_subset["RECENTLY_MARRIED"] = np.where(
    (units_subset["MARRINYR_REF"] == 2) & (units_subset["MARRINYR_SEC"] == 2), 1, 0
)

units_subset["MARRIED_MORE_THAN_YEAR"] = np.where(
    units_subset["MARRIED"] & ~units_subset["RECENTLY_MARRIED"], 1, 0
)

In [ ]:
units_subset["IN_MILITARY_REF"] = np.where(units_subset["VETSTATD_REF"] == 12, 1, 0)
units_subset["IN_MILITARY_SEC"] = np.where(units_subset["VETSTATD_SEC"] == 12, 1, 0)
units_subset["IN_MILITARY"] = np.where(
    (units_subset["IN_MILITARY_REF"] == 1) | (units_subset["IN_MILITARY_SEC"] == 1),
    1,
    0,
)
units_subset["UNEMPLOYED"] = np.where(
    (units_subset["EMPSTAT_REF"] == 2) & (units_subset["EMPSTAT_SEC"] == 2), 1, 0
)
units_subset["IN_LABOR_FORCE"] = np.where(
    units_subset["EMPSTAT_REF"].isin([1, 2]) | units_subset["EMPSTAT_SEC"].isin([1, 2]),
    1,
    0,
)
units_subset["NOT_IN_LABOR_FORCE"] = np.where(units_subset["IN_LABOR_FORCE"] == 1, 0, 1)

In [ ]:
RACE_GROUPS = {
    "WHITE": [1],
    "BLACK": [2],
    "INDIAN": [3],
    "AAPI": [4, 5, 6],
    "OTHER_RACE": [7, 8, 9],
}

for suffix in CATEGORIES:
    race = units_subset[f"RACE_{suffix}"]
    hisp = units_subset[f"HISPAN_{suffix}"]

    latino = hisp.ne(0) & hisp.ne(9)  # 9 = not reported, absent in 2018
    units_subset[f"LATINO_{suffix}"] = latino.astype(int)

    # Hispanic takes precedence, so the six categories stay mutually exclusive
    # and line up with the SE_B04001 area shares (non-Hispanic race + Hispanic).
    for name, codes in RACE_GROUPS.items():
        units_subset[f"{name}_{suffix}"] = (race.isin(codes) & ~latino).astype(int)

    units_subset[f"RACE_ETHNICITY_{suffix}"] = np.where(latino, 99, race)

In [ ]:
units_subset.to_parquet(f"pums_{year}.parquet", compression="gzip")

In [ ]:
# # Use SFRELATE if present, otherwise RELATE
# relate_col = "SFRELATE" if "SFRELATE" in df.columns else "RELATE"

# df["IS_SECONDARY"] = ((df["SFRELATE"] == 2) | (df["RELATE"] == 2)).astype(int)

# # Count per UNIT
# secondary_counts = (
#     df.groupby("UNIT", sort=False)["IS_SECONDARY"]
#     .sum()
#     .reset_index(name="N_SECONDARY")
# )
# print(secondary_counts["N_SECONDARY"].describe())
# print(((df["SFRELATE"] == 2) + (df["RELATE"] == 2)).max())

In [ ]:
# people who've recently had children category
# NOTE: this is a little iffy since this only applies to the women

# def add_recent_child_flag(df: pd.DataFrame) -> pd.DataFrame:
#     """FER_CL cleaning and inference for whether a household recently had a child."""
#     df["FER_CL"] = df["FER"].fillna(0)
#     df["FER_CL"] = np.where(df["FER_CL"] == 2, 0, df["FER_CL"])
#     rec_child = df.groupby("SERIALNO")["FER_CL"].max()
#     df["REC_CHILD"] = rec_child.loc[df["SERIALNO"]].values
#     return df
# df = lclean.add_recent_child_flag(df)
# df["FER_CL"].value_counts()